# Dzongkha-English NLLB-200 Fine-Tuning Notebook
Fine-tuning facebook/nllb-200-distilled-600M with custom Dzongkha-English corpus

## 1. Install & Import Requirements

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate torch huggingface_hub
!pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 30.5 MB/s eta 0:00:00


In [ ]:
!pip install -q -U datasets
!pip install -q evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.0 MB/s eta 0:00:00


In [ ]:
import os
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from huggingface_hub import notebook_login
import evaluate
import numpy as np

In [ ]:
os.environ["WANDB_API_KEY"] = os.environ.get("WANDB_API_KEY", "")  # read from the environment; never hardcode a key
os.environ["WANDB_PROJECT"] = "nllb-dz-en-translation"  # Set your project name
os.environ["WANDB_LOG_MODEL"] = "checkpoint"  # To log model checkpoints

## 2. Login & Configuration

In [ ]:
# Hugging Face Login
notebook_login()

In [ ]:
# Configuration
MODEL_NAME = "facebook/nllb-200-distilled-600M"
DATASET_NAME = "kinleyrabgay/dz_to_en"  # Replace with your dataset
OUTPUT_DIR = "nllb-600m-dz-en-checkpoints"
REPO_NAME = "nllb-600m-dz-en"  # For HF Hub

# Language codes
SRC_LANG = "dzo"
TGT_LANG = "eng"

## 3. Load Dataset & Model

In [ ]:
# # Load dataset
# dataset = load_dataset(DATASET_NAME)
# print(f"Dataset structure: {dataset}")

# # Create validation split if not exists
# if "validation" not in dataset:
#     dataset = dataset["train"].train_test_split(test_size=0.1)
#     dataset["validation"] = dataset.pop("test")

In [ ]:
# Load dataset
corpus = load_dataset(DATASET_NAME)
print(f"Original dataset structure: {corpus}")

# Define your index limits for each split
TRAIN_START, TRAIN_END = 0, 2000       # First 2000 training examples
VAL_START, VAL_END = 0, 500           # First 500 validation examples
TEST_START, TEST_END = 0, 500         # First 500 test examples

# Create limited splits while preserving original structure
dataset = DatasetDict({
    "train": corpus["train"].select(range(TRAIN_START, TRAIN_END)),
    "validation": corpus["validation"].select(range(VAL_START, VAL_END)),
    "test": corpus["test"].select(range(TEST_START, TEST_END))
})

# If no validation exists, create it from limited train
if "validation" not in dataset:
    dataset = dataset["train"].select(range(TRAIN_START, TRAIN_END))
    split = dataset.train_test_split(test_size=0.1)
    dataset = DatasetDict({
        "train": split["train"],
        "validation": split["test"],  # Our validation set
        "test": dataset["test"].select(range(TEST_START, TEST_END))
    })

print(f"\nLimited dataset structure: {dataset}")
print(f"Train: {len(dataset['train'])} examples (indices {TRAIN_START}-{TRAIN_END-1})")
print(f"Validation: {len(dataset['validation'])} examples (indices {VAL_START}-{VAL_END-1})")
print(f"Test: {len(dataset['test'])} examples (indices {TEST_START}-{TEST_END-1})")

README.md:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/15.9M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/244k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/248k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/225565 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3488 [00:00<?, ? examples/s]

Original dataset structure: DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 225565
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 3436
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 3488
    })
})

Limited dataset structure: DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 500
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 500
    })
})
Train: 2000 examples (indices 0-1999)
Validation: 500 examples (indices 0-499)
Test: 500 examples (indices 0-499)


In [ ]:
# Load model and tokenizer
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="dzo",
    tgt_lang="eng",
    use_fast=False
)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

## 3.5 Inspect Dataset Structure


In [ ]:
# First let's examine the dataset structure
print("Dataset features:", dataset["train"].features)
print("\nFirst example:", dataset["train"][0])

# Identify the correct column names for Dzongkha and English
dz_column = None
en_column = None

# Try to automatically detect column names
for column in dataset["train"].column_names:
    if column.lower() in ["dz", "dzo", "dzongkha", "source"]:
        dz_column = column
    elif column.lower() in ["en", "eng", "english", "target"]:
        en_column = column

# If automatic detection fails, set them manually
if not dz_column or not en_column:
    dz_column = "dz"  # Replace with your actual Dzongkha column name
    en_column = "en"  # Replace with your actual English column name

print(f"\nUsing columns: Dzongkha='{dz_column}', English='{en_column}'")

Dataset features: {'translation': {'dz': Value(dtype='string', id=None), 'en': Value(dtype='string', id=None)}}

First example: {'translation': {'dz': 'འ་ནཱི་ འདི་ བདེན་པ་ར་ མེན་ པས ', 'en': 'it is just not right'}}

Using columns: Dzongkha='dz', English='en'


## 4. Preprocessing

In [ ]:
def preprocess_function(examples):
    processed_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    for translation in examples["translation"]:
        try:
            # Extract and clean texts
            dz_text = str(translation.get("dz", "")).strip()
            en_text = str(translation.get("en", "")).strip()

            if not dz_text or not en_text:
                continue

            # Tokenize source with source language code
            inputs = tokenizer(
                f">>{SRC_LANG}<< {dz_text}",
                max_length=128,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            # Tokenize target with target language code (modern approach)
            labels = tokenizer(
                text_target=f">>{TGT_LANG}<< {en_text}",
                max_length=128,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            # Verify we got valid tokenization
            if inputs["input_ids"].numel() == 0 or labels["input_ids"].numel() == 0:
                print(f"Skipping empty tokenization for: {translation}")
                continue

            processed_batch["input_ids"].append(inputs["input_ids"][0].tolist())
            processed_batch["attention_mask"].append(inputs["attention_mask"][0].tolist())
            processed_batch["labels"].append(labels["input_ids"][0].tolist())

        except Exception as e:
            print(f"Skipping problematic example: {translation}")
            print(f"Error details: {str(e)}")
            continue

    return processed_batch

In [ ]:
# First test with a small subset
test_dataset = dataset["train"].select(range(100))
test_tokenized = test_dataset.map(
    preprocess_function,
    batched=True,
    batch_size=10,
    remove_columns=dataset["train"].column_names
)

print(f"Test run successful with {len(test_tokenized)} examples")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Test run successful with 100 examples


In [ ]:
# Now process full dataset
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=100,
    remove_columns=dataset["train"].column_names,
    num_proc=4
)

# Filter and save
tokenized_datasets = tokenized_datasets.filter(
    lambda x: len(x["input_ids"]) > 0
)

print(f"Final dataset contains {len(tokenized_datasets['train'])} examples")

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Final dataset contains 2000 examples


In [ ]:
# 1. Check dataset structure
print("Dataset structure:", tokenized_datasets)
print("\nFeatures:", tokenized_datasets["train"].features)

# 2. Examine first few examples
print("\nFirst example:")
print(tokenized_datasets["train"][0])

# 3. Check tokenization of sample text
sample_dz = "ག་ནི་བ་ ཡིད་ཕྲོག་"
sample_en = "what a lovely"

print("\nSample Dzongkha text:", sample_dz)
print("Tokenized Dzongkha:", tokenizer.tokenize(sample_dz))
print("Token IDs:", tokenizer(sample_dz)["input_ids"])

print("\nSample English text:", sample_en)
print("Tokenized English:", tokenizer.tokenize(sample_en))
print("Token IDs:", tokenizer(sample_en)["input_ids"])

# 4. Verify detokenization
print("\nDetokenized example:")
print("Input:", tokenizer.decode(tokenized_datasets["train"][0]["input_ids"]))
print("Label:", tokenizer.decode(tokenized_datasets["train"][0]["labels"]))

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 500
    })
})

Features: {'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None), 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None), 'labels': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None)}

First example:
{'input_ids': [3, 20545, 248072, 708, 248124, 248124, 143415, 371, 21519, 92234, 141106, 802, 7359, 4879, 62191, 248234, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

## 5. Training Setup

In [ ]:
# Data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=tokenizer.pad_token_id
)

In [ ]:
!pip install -q sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 368.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 492.8 kB/s eta 0:00:00


In [ ]:
# Metrics
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [ ]:
# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,  # Keep last 3 checkpoints
    num_train_epochs=5,
    predict_with_generate=True,
    fp16=True,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    push_to_hub=True,
    hub_model_id=f"kinleyrabgay/{REPO_NAME}",
    hub_strategy="checkpoint",  # Push checkpoints during training
    report_to="none",  # Optional
)

In [ ]:
# # First verify GPU
# import torch
# assert torch.cuda.is_available(), "No GPU detected in Colab!"
# print(f"GPU: {torch.cuda.get_device_name(0)}")
# print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

# # Optimized training arguments for Colab GPU
# training_args = Seq2SeqTrainingArguments(
#     output_dir=OUTPUT_DIR,
#     overwrite_output_dir=True,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=3e-5,
#     per_device_train_batch_size=4,  # Conservative for Colab GPUs
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=2,  # Effective batch size of 8
#     weight_decay=0.01,
#     save_total_limit=2,  # Save space on Colab
#     num_train_epochs=5,
#     predict_with_generate=True,
#     fp16=True,  # Enable mixed precision
#     logging_dir="./logs",
#     load_best_model_at_end=True,
#     metric_for_best_model="bleu",
#     greater_is_better=True,
#     push_to_hub=True,
#     hub_model_id=f"kinleyrabgay/{REPO_NAME}",
#     hub_strategy="checkpoint",
#     hub_token=os.getenv('HF_TOKEN'),
#     dataloader_pin_memory=True,
#     dataloader_num_workers=2,  # Don't exceed Colab's CPU cores
#     report_to="none",  # Disable WandB to save resources
# )

# # Clear GPU cache
# torch.cuda.empty_cache()

AssertionError: No GPU detected in Colab!

In [ ]:
# Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# 2. Initialize trainer with processing_class
# trainer = Seq2SeqTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_datasets["train"],
#     eval_dataset=tokenized_datasets["validation"],
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
#     # New way to handle tokenization:
#     tokenizer=tokenizer  # Still works but will be deprecated
#     # Future alternative (when available):
#     # processing_class=YourCustomProcessorClass
# )

<ipython-input-22-4063214741>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


## 6. Training


In [ ]:
# Start training
train_result = trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


In [ ]:
# Save final model and metrics
trainer.save_model(OUTPUT_DIR)
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

## 7. Push to Hub & Create Model Card


In [ ]:
# Create model card
model_card = f"""
---
language:
- {SRC_LANG}
- {TGT_LANG}
tags:
- translation
- nllb
license: cc-by-nc-4.0
datasets:
- {DATASET_NAME}
metrics:
- bleu
---

0M Fine-tuned for Dzongkha-English Translation

This model is a fine-tuned version of {MODEL_NAME} on {DATASET_NAME}.

## Training Results

| Epoch | BLEU Score |
|-------|------------|
| 1     | {metrics.get('eval_bleu', {}).get(0, 'N/A')} |
| 2     | {metrics.get('eval_bleu', {}).get(1, 'N/A')} |
| 3     | {metrics.get('eval_bleu', {}).get(2, 'N/A')} |
"""

with open(os.path.join(OUTPUT_DIR, "README.md"), "w") as f:
    f.write(model_card)

# Push model card
trainer.push_to_hub(commit_message="Add model card")

## 8. Load Checkpoints (Example)


In [ ]:
# Example: Load specific checkpoint
checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoint-500")  # Example checkpoint

checkpoint_model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)
checkpoint_tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)


## 9. Inference Example


In [ ]:
from transformers import pipeline

# Load best model (from final training)
translator = pipeline(
    "translation",
    model=f"your_username/{REPO_NAME}",
    device=0 if torch.cuda.is_available() else -1
)

# Example translation
dz_text = "ག་ནི་བ་ ཡིད་ཕྲོག་ པའི་ རྩོམ་བྲི་པ་ ཅིག་ ཨིན་ མས"
translation = translator(
    dz_text,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG
)
print(f"Input: {dz_text}")
print(f"Translation: {translation[0]['translation_text']}")